# M1b adım adım: ne ekledik, ne işe yaradı

İlk defter (`rag_walkthrough_tr.ipynb`) tek bir soruyu baseline RAG'in her aşamasından geçirdi. Bu defter **M1b'de eklenen her parçayı** aynı soru üzerinde gerçek çıktılarıyla gösterir ve 250 soruluk deney sonuçlarını açıklar:

0. Kurulum → 1. Sonuç tablosu (+ güven aralıkları) → 2. Bölüm chunking → 3. Paragraf atfı → 4. Evidence-F1 tavanı → 5. Reranker → 6. Multi-query / HyDE → 7. Docling PDF + kanıt hizalama → 8. LLM-as-judge → 9. Soru bazlı karşılaştırma

**Kısa cevap:** M2'ye taşınacak aday pipeline = baseline + **bölüm chunking** (`chunking.strategy: section`) + **paragraf atfı** (`generation.cite_unit: paragraph`). Geri kalan denemeler (reranker, multi-query, HyDE, Docling, prompt_v2) ölçüldü ve elendi; nedenleri aşağıda.

**Nasıl kullanılır:** hücreleri sırayla çalıştır (`Shift+Enter`). Büyük harfli değişkenler (`QUESTION_INDEX`, `RUNS`…) senin değiştirmen için. **Gerekenler:** GPU (bge-m3, reranker) ve 3, 6, 8. adımlar için Ollama. Tüm kod `src/paperlens/` içindeki gerçek proje kodudur.

## 0 · Kurulum

İki config yüklüyoruz: `baseline.yaml` (M1a'nın sonucu) ve `m1b_section_best.yaml` (M1b adayı). Aday, baseline'ı `extends:` ile miras alır ve yalnızca iki alanı değiştirir; hücre bu farkı config'in kendisinden çıkarır.

In [1]:
import json
import os
from pathlib import Path

# Defter notebooks/ klasöründe; config'deki göreli yollar (data/, prompts/) proje kökünden çalışır
if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)

from paperlens.config.loader import load_config

base_cfg = load_config("experiments/configs/baseline.yaml")
best_cfg = load_config("experiments/configs/m1b_section_best.yaml")

print("baseline -> aday arasında değişen ayarlar:")
for section in ("data", "chunking", "retrieval", "rerank", "query", "generation"):
    a = getattr(base_cfg, section).model_dump()
    b = getattr(best_cfg, section).model_dump()
    for key in a:
        if a[key] != b[key]:
            print(f"  {section}.{key}: {a[key]!r} -> {b[key]!r}")

baseline -> aday arasında değişen ayarlar:
  chunking.strategy: 'fixed' -> 'section'
  chunking.chunk_overlap: 40 -> 0
  generation.cite_unit: 'chunk' -> 'paragraph'


## 1 · Sonuç tablosu

Her deney baseline'dan **tek bir şeyi** değiştirdi ve aynı 250 soruda koştu (`evals/results/<koşu>/metrics.json`). Sütunlar:

- **answer_f1**: cevap kelimelerinin gold ile örtüşmesi (resmî QASPER metriği).
- **judge**: yerel bir LLM'in (qwen3:14b) "bu cevap doğru mu?" kararı; F1'in uzun ama doğru cevaplara haksızlığını dengeler (8. adım).
- **ev_f1 / tavan**: atıf yapılan paragrafların gold kanıtla örtüşmesi ve bu retrieval ile ulaşılabilecek en iyi değer (4. adım).
- **R@5**: ilk 5 chunk'tan en az biri gold paragraf içeriyor mu.
- **red_rec**: cevapsız sorularda modelin gerçekten "cevaplanamaz" deme oranı.
- **p95_ms**: soruların %95'inin bittiği süre.

In [2]:
RESULTS = Path("evals/results")
RUNS = [  # <- koşu ekle/çıkar (evals/results altındaki klasör adları)
    "baseline",
    "prompt_v2",
    "rerank",
    "hybrid_rerank",
    "multi_query",
    "hyde",
    "cite_paragraph",
    "chunk_semantic",
    "chunk_section_200",
    "source_docling",
    "m1b_best",
    "m1b_section_best",
]
COLS = {
    "answer_f1": "answer_f1",
    "judge.accuracy": "judge",
    "evidence_f1": "ev_f1",
    "evidence_f1_ceiling": "tavan",
    "retrieval.recall_at_5": "R@5",
    "refusal_recall": "red_rec",
    "latency_p95_ms": "p95_ms",
}
metrics = {r: json.loads((RESULTS / r / "metrics.json").read_text()) for r in RUNS}
print(f"{'koşu':18}" + "".join(f"{c:>10}" for c in COLS.values()))
for run, m in metrics.items():
    row = "".join(
        f"{m.get(k, float('nan')):>10.0f}"
        if k.endswith("_ms")
        else f"{m.get(k, float('nan')):>10.3f}"
        for k in COLS
    )
    print(f"{run:18}{row}")

koşu               answer_f1     judge     ev_f1     tavan       R@5   red_rec    p95_ms
baseline               0.419     0.500     0.297     0.524     0.703     0.270      1851
prompt_v2              0.421     0.528     0.292     0.524     0.703     0.216      1624
rerank                 0.387     0.496     0.285     0.526     0.731     0.270      2897
hybrid_rerank          0.387     0.496     0.285     0.526     0.731     0.270      2905
multi_query            0.427     0.496     0.271     0.505     0.672     0.216      2917
hyde                   0.414     0.484     0.273     0.529     0.713     0.270      4124
cite_paragraph         0.421     0.524     0.455     0.787     0.703     0.297      1809
chunk_semantic         0.427     0.548     0.413     0.658     0.667     0.297      1501
chunk_section_200      0.442     0.556     0.425     0.664     0.665     0.243      1451
source_docling         0.404     0.532     0.253     0.484     0.660     0.189      3339
m1b_best             

### Bu farklar gerçek mi? Paired bootstrap

250 soruda ±0.02'lik fark şans eseri çıkabilir. `evals/stats.py` → `paired_bootstrap()` iki koşunun **aynı sorulardaki** farklarını 2000 kez yeniden örnekler ve ortalama farkın %95 güven aralığını (GA) verir. **GA 0'ı içeriyorsa** "bu örneklemle kazanç kanıtlanmadı" demektir.

Aynı testi hem answer_f1 hem evidence_f1 için yapıyoruz: M1b'nin asıl dersi bu ikisinin farkında.

In [3]:
from paperlens.evals.report import load_run
from paperlens.evals.stats import paired_bootstrap

runs = {r: load_run(RESULTS / r) for r in RUNS}
base = runs["baseline"].records


def by_question(records, field):
    return {r.question_id: getattr(r, field) for r in records}


print(f"{'koşu':18}{'Δ answer_f1 [%95 GA]':>28}{'Δ evidence_f1 [%95 GA]':>28}")
for run in RUNS[1:]:
    parts = []
    for field in ("answer_f1", "evidence_f1"):
        res = paired_bootstrap(by_question(runs[run].records, field), by_question(base, field))
        mark = " *" if res.low > 0 or res.high < 0 else "  "  # * = GA 0'ı dışarıda bırakıyor
        parts.append(f"{res.mean_diff:+.3f} [{res.low:+.3f}, {res.high:+.3f}]{mark}")
    print(f"{run:18}{parts[0]:>28}{parts[1]:>28}")
print("\n* = güven aralığı 0'ı içermiyor (fark bu örneklemde anlamlı)")

koşu                      Δ answer_f1 [%95 GA]      Δ evidence_f1 [%95 GA]
prompt_v2            +0.003 [-0.019, +0.026]     -0.005 [-0.023, +0.013]  
rerank               -0.032 [-0.069, +0.005]     -0.012 [-0.043, +0.018]  


hybrid_rerank        -0.032 [-0.069, +0.005]     -0.012 [-0.043, +0.018]  


multi_query          +0.008 [-0.027, +0.045]     -0.026 [-0.063, +0.010]  
hyde                 -0.005 [-0.034, +0.025]     -0.024 [-0.050, +0.000]  
cite_paragraph       +0.002 [-0.018, +0.023]     +0.158 [+0.120, +0.198] *
chunk_semantic       +0.009 [-0.030, +0.045]     +0.116 [+0.071, +0.160] *
chunk_section_200    +0.024 [-0.013, +0.058]     +0.128 [+0.079, +0.172] *
source_docling       -0.015 [-0.057, +0.025]     -0.044 [-0.083, -0.007] *
m1b_best             +0.016 [-0.024, +0.053]     +0.173 [+0.122, +0.224] *
m1b_section_best     +0.022 [-0.015, +0.056]     +0.194 [+0.144, +0.242] *

* = güven aralığı 0'ı içermiyor (fark bu örneklemde anlamlı)


**Okuma:** answer_f1 farklarının hiçbiri anlamlı değil, yani hiçbir değişiklik cevap doğruluğunu kanıtlanmış biçimde artırmadı. Kanıt gösterme (evidence_f1) ise chunking ve paragraf atfıyla açıkça arttı; asıl kazanç orada. Retrieval'ı değiştiren denemeler (rerank, multi_query, HyDE) evidence_f1'i hiç artırmadı ya da düşürdü.

Dikkat: 20+ karşılaştırma yapınca birkaçı şans eseri "anlamlı" çıkabilir (çoklu karşılaştırma). Bu yüzden tek bir yıldıza değil, büyüklüğü ve tutarlılığı yüksek farklara güveniyoruz.

## 2 · Veri ve soru

İlk defterdeki SnapCaptions sorusu (16). Bu soruda baseline doğru cevabı verdi ama kanıt skoru 0.33 idi; adayda cevap aynı, kanıt skoru 1.0. Farkın nereden geldiğini adım adım göreceğiz.

In [4]:
from paperlens.ingestion.qasper import download_split, load_split, sample_questions

papers, questions = load_split(download_split(base_cfg.data, base_cfg.data.split))
sample = sample_questions(questions, base_cfg.data.sample_size, base_cfg.data.seed)

QUESTION_INDEX = 16  # <- değiştir (0-249); 5, 21, 45 de kanıtı belirgin artan örnekler
question = sample[QUESTION_INDEX]
paper = next(p for p in papers if p.paper_id == question.paper_id)
gold_ids = {pid for r in question.references for pid in r.evidence_ids}
para_of = {p.paragraph_id: p for p in paper.paragraphs}

print("Soru  :", question.question)
print("Makale:", paper.paper_id, "-", paper.title, f"({len(paper.paragraphs)} paragraf)")
for i, r in enumerate(question.references, 1):
    print(f"Gold {i} ({r.answer_type}): {r.answer_text}  | kanıt: {r.evidence_ids}")

Soru  : How large is their MNER SnapCaptions dataset?
Makale: 1802.07862 - Multimodal Named Entity Recognition for Short Social Media Posts (69 paragraf)
Gold 1 (extractive): 10K user-generated image (snap) and textual caption pairs  | kanıt: ['1802.07862::s6p0']
Gold 2 (abstractive): 10000  | kanıt: ['1802.07862::s6p0']


## 3 · Bölüm chunking (`chunking.strategy: section`)

`ingestion/chunkers.py` → `chunk_paper()` üç strateji sunar:

| strateji | nasıl böler | sonuç |
|---|---|---|
| `fixed` (baseline) | tüm metni 200 kelimelik pencerelerle keser, 40 kelime örtüşür | paragraflar ortadan bölünür, bir paragraf iki chunk'ta tekrar eder |
| `section` (aday) | **bütün paragrafları** 200 kelimeye kadar aynı chunk'a doldurur, **bölüm sınırını asla geçmez** | her chunk tam paragraflardan oluşur; tek paragraf 200 kelimeden uzunsa yalnız o bölünür |
| `semantic` (M1b'de eklendi) | cümleleri embed eder, ardışık iki cümlenin anlamı uzaklaşınca keser | anlam sınırında böler ama her cümleyi embed etmek gerekir |

`section` M1a'da da vardı; M1b'de semantic ile karşılaştırınca daha iyi ve daha basit olduğu ortaya çıktı. Aşağıda bu makale için üçünü yan yana görüyoruz.

In [5]:
from collections import Counter

from paperlens.index.embedder import build_embedder
from paperlens.ingestion.chunkers import chunk_paper

embedder = build_embedder(base_cfg.embedding)  # semantic strateji cümleleri embed etmek için ister
STRATEGIES = {
    "fixed (baseline)": base_cfg.chunking,
    "section (aday)": best_cfg.chunking,
    "semantic": load_config("experiments/configs/m1b/chunk_semantic.yaml").chunking,
}
chunks_by = {name: chunk_paper(paper, c, embedder) for name, c in STRATEGIES.items()}

print(f"{'strateji':18}{'chunk':>7}{'ort. kelime':>13}{'min':>6}{'max':>6}{'bölünen paragraf':>18}")
for name, chunks in chunks_by.items():
    lens = [len(c.text.split()) for c in chunks]
    seen = Counter(pid for c in chunks for pid in c.paragraph_ids)
    split = sum(n > 1 for n in seen.values())  # birden fazla chunk'a dağılan paragraf
    print(
        f"{name:18}{len(chunks):>7}{sum(lens) / len(lens):>13.0f}{min(lens):>6}{max(lens):>6}{split:>18}"
    )

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Batches:   0%|          | 0/6 [00:00<?, ?it/s]

strateji            chunk  ort. kelime   min   max  bölünen paragraf
fixed (baseline)       27          197    69   206                43
section (aday)         30          143     4   203                 2
semantic               32          134    40   202                14


Şimdi gold paragrafı içeren chunk'a bakalım. Her chunk `segments` listesi taşır: içindeki her paragraf parçası ve o parçanın paragraf id'si (M1b'de eklendi, paragraf atfı bunu kullanır). `fixed`'de chunk bir paragrafın ortasında başlayıp başka bir paragrafın ortasında biter; `section`'da sınırlar paragraf sınırlarıyla aynıdır.

In [6]:
for name in ("fixed (baseline)", "section (aday)"):
    chunk = next(c for c in chunks_by[name] if gold_ids & set(c.paragraph_ids))
    print(
        f"=== {name}: {chunk.chunk_id}  ({len(chunk.text.split())} kelime, bölüm: {chunk.section})"
    )
    for seg in chunk.segments:
        full = len(seg.text.split()) == len(para_of[seg.paragraph_id].text.split())
        mark = "GOLD" if seg.paragraph_id in gold_ids else "    "
        print(
            f"  {mark} {seg.paragraph_id:>45}  {'tam   ' if full else 'PARÇA '} {seg.text[:70]!r}..."
        )
    print()

=== fixed (baseline): 1802.07862::c15  (202 kelime, bölüm: Modality Attention)
                                    1802.07862::s5p7  PARÇA  ')'...
                                    1802.07862::s5p8  tam    "t(m) = (at(m))m'{w,c}(at(m')) m {w,c}"...
                                    1802.07862::s5p9  tam    'xt = m{w,c} t(m)xt(m)'...
                                   1802.07862::s5p10  tam    'Note that while we apply this modality attention module to the Bi-LSTM'...
  GOLD                              1802.07862::s6p0  PARÇA  'The SnapCaptions dataset is composed of 10K user-generated image (snap'...

=== section (aday): 1802.07862::c17  (199 kelime, bölüm: SnapCaptions Dataset)
  GOLD                              1802.07862::s6p0  tam    'The SnapCaptions dataset is composed of 10K user-generated image (snap'...



## 4 · Paragraf atfı (`generation.cite_unit: paragraph`)

Model cevabın kanıtını numarayla gösterir. Kanıt skoru (Evidence-F1), atıf yapılan **paragraf id'lerinin** gold paragraflarla örtüşmesidir.

- **`chunk` (baseline):** prompt'taki her chunk tek numara alır: `[1]`, `[2]`… Model `[1]` dediğinde o chunk'taki **bütün paragraflar** kanıt sayılır. Chunk'ta 3 paragraf varsa ve yalnız biri gold'sa precision 1/3 olur, model mükemmel seçse bile.
- **`paragraph` (aday):** `pipelines/baseline.py` → `format_paragraph_context()` her chunk'ın içindeki **her paragraf parçasını** ayrı numaralar. Model tam olarak kanıt olan paragrafı gösterebilir.

Retrieval değişmez: aynı chunk'lar, aynı sıra. Değişen yalnızca prompt'taki numaralandırma ve atfın paragraf id'lerine çevrilmesi.

Üç kurulumu karşılaştırıyoruz; her biri bir deney koşusuna denk gelir:

| kurulum | chunking | atıf | koşu |
|---|---|---|---|
| baseline | fixed | chunk | `baseline` |
| sadece paragraf atfı | fixed | paragraph | `m1b/cite_paragraph` |
| aday | section | paragraph | `m1b_section_best` |

İlk ikisi aynı (fixed) indeksi kullanır, aday kendi (section) indeksini.

In [7]:
from paperlens.index.store import build_or_load, index_key
from paperlens.llm.client import OpenAICompatClient
from paperlens.pipelines.factory import build_pipeline

SETUPS = {  # ad -> config; indeks, config'in chunking ayarından bulunur (hepsi diskte hazır)
    "baseline": base_cfg,
    "sadece paragraf atfı": load_config("experiments/configs/m1b/cite_paragraph.yaml"),
    "aday": best_cfg,
}
bundles = {}
for name, c in SETUPS.items():
    key = index_key(c, papers)
    if key not in bundles:
        bundles[key] = build_or_load(c, papers, embedder)
    print(
        f"{name:22} indeks={key} chunk={len(bundles[key].chunks)} cache hit={bundles[key].cache_hit}"
    )
bundle = bundles[index_key(best_cfg, papers)]  # sonraki adımlar adayın indeksini kullanır

titles = {p.paper_id: p.title for p in papers}
llm = OpenAICompatClient(best_cfg.llm)
pipes = {
    name: build_pipeline(c, bundles[index_key(c, papers)], llm, titles)
    for name, c in SETUPS.items()
}

2026-10-21 02:08:24 [info     ] index cache hit                chunks=6676 key=b6e671ce5a26


baseline               indeks=b6e671ce5a26 chunk=6676 cache hit=True
sadece paragraf atfı   indeks=b6e671ce5a26 chunk=6676 cache hit=True


2026-10-21 02:08:25 [info     ] index cache hit                chunks=8360 key=0fdafd6f7ccb


aday                   indeks=0fdafd6f7ccb chunk=8360 cache hit=True


Gold paragrafı içeren chunk prompt'ta nasıl görünüyor? Baseline'da tek bir numara bütün chunk'ı (5 paragraf parçası) temsil eder; paragraf atfında her parça kendi numarasını alır. (Numaralar burada yalnız bu chunk için 1'den başlıyor; gerçek prompt'ta 5 chunk boyunca devam eder.)

In [8]:
from paperlens.pipelines.baseline import format_context, format_paragraph_context

for name in ("baseline", "sadece paragraf atfı"):
    retrieved = pipes[name].retrieve(question, pipes[name].plan(question))
    gold_hits = [sc for sc in retrieved if gold_ids & set(sc.chunk.paragraph_ids)]
    if not gold_hits:
        print(f"===== {name}: gold paragraf ilk 5 chunk'ta yok\n")
        continue
    if SETUPS[name].generation.cite_unit == "paragraph":
        text = format_paragraph_context(gold_hits[:1])[0]
    else:
        text = format_context(gold_hits[:1])
    print(f"===== {name}: {gold_hits[0].chunk.chunk_id} prompt'ta =====")
    for line in text.splitlines():
        print("  " + (line[:150] + "..." if len(line) > 150 else line))
    print()

===== baseline: 1802.07862::c15 prompt'ta =====
  [1] (section: Modality Attention)
  [Modality Attention] ) t(m) = (at(m))m'{w,c}(at(m')) m {w,c} xt = m{w,c} t(m)xt(m) Note that while we apply this modality attention module to the Bi-L...



===== sadece paragraf atfı: 1802.07862::c15 prompt'ta =====
  Passage from section: Modality Attention
  [1] )
  [2] t(m) = (at(m))m'{w,c}(at(m')) m {w,c}
  [3] xt = m{w,c} t(m)xt(m)
  [4] Note that while we apply this modality attention module to the Bi-LSTM+CRF architecture (Section SECREF4 ) for its empirical superiority, the modu...
  [5] The SnapCaptions dataset is composed of 10K user-generated image (snap) and textual caption pairs where named entities in captions are manually la...



Şimdi üç kurulumda da LLM'i çalıştırıp atfın hangi paragraf id'lerine çevrildiğine ve kanıt skoruna bakıyoruz (Ollama gerekir).

In [9]:
from paperlens.evals.records import build_record

recs = {}
for name, pipe in pipes.items():
    result = pipe.run(question)
    rec = build_record(question, result, best_cfg.eval.k_values, SETUPS[name].generation.cite_unit)
    recs[name] = rec
    a = result.answer
    print(f"===== {name}")
    print("cevap       :", a.text)
    print("atıf (chunk):", a.citations)
    print("kanıt id    :", a.evidence_ids)
    print("gold id     :", sorted(gold_ids))
    print(
        f"answer_f1={rec.answer_f1:.3f}  evidence_f1={rec.evidence_f1:.3f}  tavan={rec.evidence_f1_ceiling:.3f}\n"
    )

===== baseline
cevap       : 10K user-generated image (snap) and textual caption pairs
atıf (chunk): ['1802.07862::c15']
kanıt id    : ['1802.07862::s5p7', '1802.07862::s5p8', '1802.07862::s5p9', '1802.07862::s5p10', '1802.07862::s6p0']
gold id     : ['1802.07862::s6p0']
answer_f1=1.000  evidence_f1=0.333  tavan=0.333



===== sadece paragraf atfı
cevap       : 10K user-generated image (snap) and textual caption pairs
atıf (chunk): ['1802.07862::c15']
kanıt id    : ['1802.07862::s6p0']
gold id     : ['1802.07862::s6p0']
answer_f1=1.000  evidence_f1=1.000  tavan=1.000



===== aday
cevap       : 10K user-generated image (snap) and textual caption pairs
atıf (chunk): ['1802.07862::c17']
kanıt id    : ['1802.07862::s6p0']
gold id     : ['1802.07862::s6p0']
answer_f1=1.000  evidence_f1=1.000  tavan=1.000



## 5 · Evidence-F1 tavanı

`evals/records.py` → `evidence_ceiling()`: getirilen chunk'larla **kusursuz atıf** yapılsaydı ulaşılabilecek en iyi Evidence-F1. İki şeyi ayırır:

- **tavan düşükse** sorun retrieval'da veya chunk boyutunda (doğru paragraf ya gelmedi ya da gürültülü bir chunk'ın içinde geldi),
- **skor tavandan çok düşükse** sorun modelin atfında.

`chunk` modunda bir atıf bütün chunk'ı kapsar, bu yüzden tavan chunk genişliğiyle sınırlıdır. `paragraph` modunda her paragraf tek başına atfedilebilir, tavan yalnızca retrieval'a bağlıdır. 250 soruda tavan .524'ten (baseline) .770'e (aday) çıktı; bu yüzden aynı model aynı chunk'larla çok daha yüksek kanıt skoru alabiliyor.

In [10]:
for run in ("baseline", "cite_paragraph", "chunk_section_200", "m1b_section_best"):
    m = metrics[run]
    print(
        f"{run:18} evidence_f1={m['evidence_f1']:.3f}  tavan={m['evidence_f1_ceiling']:.3f}  "
        f"tavanın %{100 * m['evidence_f1'] / m['evidence_f1_ceiling']:.0f}'i kullanılıyor"
    )

baseline           evidence_f1=0.297  tavan=0.524  tavanın %57'i kullanılıyor
cite_paragraph     evidence_f1=0.455  tavan=0.787  tavanın %58'i kullanılıyor
chunk_section_200  evidence_f1=0.425  tavan=0.664  tavanın %64'i kullanılıyor
m1b_section_best   evidence_f1=0.491  tavan=0.770  tavanın %64'i kullanılıyor


## 6 · Denenip elenenler: reranker

`retrieval/reranker.py` → `CrossEncoderReranker` (bge-reranker-v2-m3). Önce dense arama `retrieval.candidate_k` = 30 aday getirir, sonra cross-encoder soru ile her adayı **birlikte** okuyup yeniden puanlar ve en iyi 5'i tutar. Recall@5 .703 → .731 arttı ama answer_f1 .419 → .387 düştü ve her soruya ~1 saniye eklendi.

`rerank` ile `hybrid_rerank` neden birebir aynı çıktı? Retrieval tek makale içinde yapılıyor (`paper_id` filtresi) ve bir makalede ortalama ~24 chunk var. 30 adaylık havuz zaten neredeyse **bütün makale**; hangi yöntemin havuzu doldurduğu fark etmiyor. Reranker'ı M2'de bütün korpusta arama yapınca yeniden denemeye değer.

In [11]:
from statistics import mean

fixed_bundle = bundles[
    index_key(base_cfg, papers)
]  # rerank deneyi baseline (fixed) chunk'larıyla koştu
per_paper = Counter(c.paper_id for c in fixed_bundle.chunks)
cand_k = load_config("experiments/configs/m1b/rerank.yaml").retrieval.candidate_k
print(
    f"makale başına chunk (fixed, baseline): ort. {mean(per_paper.values()):.1f}, "
    f"{sum(n <= cand_k for n in per_paper.values())}/{len(per_paper)} makale <= candidate_k={cand_k}"
)
print(f"bu makale: {per_paper[paper.paper_id]} chunk")

makale başına chunk (fixed, baseline): ort. 23.8, 234/281 makale <= candidate_k=30
bu makale: 27 chunk


In [12]:
import gc

import torch

from paperlens.retrieval.reranker import build_reranker

reranker = build_reranker(load_config("experiments/configs/m1b/rerank.yaml").rerank)
pool = fixed_bundle.dense.search(question.question, cand_k, question.paper_id)
reranked = reranker.rerank(question.question, pool, 5)
dense_rank = {sc.chunk.chunk_id: i for i, sc in enumerate(pool, 1)}

print("reranker'ın ilk 5'i (dense sırası -> yeni sıra):")
for i, sc in enumerate(reranked, 1):
    gold = "GOLD" if gold_ids & set(sc.chunk.paragraph_ids) else "    "
    print(
        f"  {dense_rank[sc.chunk.chunk_id]:>2} -> {i}  {gold}  skor={sc.score:+.2f}  {sc.chunk.text[:70]!r}"
    )

del reranker
gc.collect()
torch.cuda.empty_cache()  # GPU'yu sonraki adımlar için boşalt

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

reranker'ın ilk 5'i (dense sırası -> yeni sıra):
   2 -> 1  GOLD  skor=+0.97  "[Modality Attention] ) t(m) = (at(m))m'{w,c}(at(m')) m {w,c} xt = m{w,"
   3 -> 2  GOLD  skor=+0.96  '[SnapCaptions Dataset] split the dataset into train (70%), validation '
   4 -> 3        skor=+0.63  '[Results: SnapCaptions Dataset] TABREF20 . By increasing the number of'
   6 -> 4        skor=+0.51  '[Introduction] target named entities, selfies, etc. Note that the prop'
   8 -> 5        skor=+0.41  '[Baselines] visual embeddings as input to entity LSTM. Table TABREF6 s'


## 7 · Denenip elenenler: multi-query ve HyDE

İkisi de `retrieval/query.py` içinde ve aramadan **önce** LLM'i çağırır. Sonuç bir `QueryPlan`: dense ve BM25'in hangi metinlerle arama yapacağı.

- **Multi-query:** LLM soruyu 3 farklı şekilde yeniden yazar; her metin ayrı aranır ve sonuçlar RRF ile birleşir.
- **HyDE:** LLM soruya **uydurma bir cevap paragrafı** yazar; dense arama bu paragrafla yapılır (gerçek paragrafa soru cümlesinden daha çok benzer diye). BM25 düz soruyla kalır, çünkü uydurma kelimeler kelime aramasını bozar.

İkisinde de kazanç çıkmadı (Recall@5 .672 ve .713), gecikme 1,5–2 kat arttı. Tek makale içinde zaten az aday olduğu için yeniden yazmanın bulacağı yeni bir şey pek yok.

In [13]:
from paperlens.retrieval.query import build_transformer

for name in ("multi_query", "hyde"):
    qcfg = load_config(f"experiments/configs/m1b/{name}.yaml").query
    p = build_transformer(qcfg, llm).plan(question.question, paper.title)
    print(f"===== {name}  (hata: {p.error})")
    for t in p.dense:
        print("  dense :", t[:200])
    for t in p.sparse:
        print("  bm25  :", t[:200])
    print()

===== multi_query  (hata: None)
  dense : How large is their MNER SnapCaptions dataset?
  dense : What is the size (number of instances) of the **SnapCaptions** dataset in **Multimodal NER** experiments?
  dense : **SnapCaptions** dataset statistics: sample count and multimodal entity annotation details from the paper?
  dense : Reported scale (instances) of the **SnapCaptions** benchmark dataset for **multimodal named entity recognition**?
  bm25  : How large is their MNER SnapCaptions dataset?
  bm25  : What is the size (number of instances) of the **SnapCaptions** dataset in **Multimodal NER** experiments?
  bm25  : **SnapCaptions** dataset statistics: sample count and multimodal entity annotation details from the paper?
  bm25  : Reported scale (instances) of the **SnapCaptions** benchmark dataset for **multimodal named entity recognition**?



===== hyde  (hata: None)
  dense : To evaluate the robustness of our proposed multimodal named entity recognition (MNER) framework, we curated the **SnapCaptions** dataset, a benchmark specifically designed for short-form social media 
  dense : How large is their MNER SnapCaptions dataset?
  bm25  : How large is their MNER SnapCaptions dataset?



## 8 · Denenip elenenler: Docling PDF + kanıt hizalama

QASPER makale metnini kendi hazır ayrıştırmasıyla verir. Gerçek hayatta elimizde PDF olur. `ingestion/pdf.py` arXiv'den PDF indirir, `ingestion/docling_parser.py` Docling ile paragraflara ayırır (`paperlens prepare-pdfs`, 278 makale ayrıştırıldı, 1 makale arXiv'de yok).

Sorun: gold kanıt QASPER paragraf id'lerine bağlı, PDF paragraflarının sınırları ve id'leri farklı. `ingestion/align.py` her gold paragrafı rapidfuzz `partial_ratio` ile PDF paragraflarında arar (eşik `data.align_min_score` = 85). QASPER'in `BIBREF12` gibi yer tutucuları önce silinir; eşleşmeyen kanıt `::unaligned` id'siyle görünür kalır ki soru "kanıtsız" gibi görünmesin.

Sonuç: gold kanıtın %90.4'ü hizalandı. Kalan ~%10 kanıt hiçbir zaman bulunamayacağı için evidence_f1 düştü (.297 → .253). Gerçekçi bir senaryo olarak değerli ama QASPER metniyle kalmak ölçüm için doğru.

In [14]:
from paperlens.ingestion.align import scored_matches
from paperlens.ingestion.docling_parser import blocks_to_paper, load_blocks
from paperlens.ingestion.pdf import parsed_path

dcfg = load_config("experiments/configs/m1b/source_docling.yaml").data
print(
    "hizalama oranı (250 soru):", round(metrics["source_docling"]["data.evidence_aligned_rate"], 3)
)
if not parsed_path(dcfg, paper.paper_id).exists():
    print("bu makalenin PDF'i ayrıştırılmamış; QUESTION_INDEX'i değiştir")
else:
    pdf_paper = blocks_to_paper(paper.paper_id, paper.title, load_blocks(dcfg, paper.paper_id))
    print(
        f"paragraf sayısı: QASPER {len(paper.paragraphs)} | Docling PDF {len(pdf_paper.paragraphs)}\n"
    )
    for gid in sorted(gold_ids):
        text = para_of[gid].text
        print(f"QASPER {gid}: {text[:110]!r}")
        hits = scored_matches(
            text,
            pdf_paper.paragraphs,
            dcfg.align_min_score,
            dcfg.align_min_words,
            dcfg.align_score_margin,
        )
        pdf_text = {p.paragraph_id: p.text for p in pdf_paper.paragraphs}
        for pid, score in hits or [("(eşleşme yok -> ::unaligned)", 0.0)]:
            print(f"  -> {pid} skor={score:.0f}  {pdf_text.get(pid, '')[:110]!r}")
        print()

hizalama oranı (250 soru): 0.904
paragraf sayısı: QASPER 69 | Docling PDF 122

QASPER 1802.07862::s6p0: 'The SnapCaptions dataset is composed of 10K user-generated image (snap) and textual caption pairs where named '
  -> 1802.07862::d38 skor=98  'The SnapCaptions dataset is composed of 10K user-generated image (snap) and textual caption pairs where named '



## 9 · LLM-as-judge

`evals/judge.py`. Answer-F1 kelime örtüşmesi sayar: "10K image-caption pairs" ile "about 10,000 pairs of images and captions" aynı bilgi ama F1 düşük. Judge, kaydedilmiş cevapları ikinci bir yerel modele (qwen3:14b) "bu cevap referanslardan biriyle aynı bilgiyi mi veriyor?" diye sorar.

Önce **kurallar**, LLM'e gitmeden karar verir (`rule_verdict()`), çünkü bu durumlar açık:
1. Bir referansla birebir eşleşme (F1 = 1) → doğru.
2. Pipeline hata verdiyse → yanlış.
3. Bütün referanslar "cevaplanamaz" ise → yalnızca model reddettiyse doğru.
4. Cevabı olan soruyu reddettiyse → yanlış. Evet/hayır sorusunda birebir eşleşme yoksa → yanlış.

Kalanlar LLM'e gider. Güven kontrolü: F1 ≥ 0.8 olan cevapların neredeyse hepsi "doğru" çıkmalı (`judge.agreement_on_high_f1`, ~0.95). Judge **ikincil** metriktir; ana metrik resmî answer_f1 olarak kalır.

In [15]:
from paperlens.evals.judge import judge_record, rule_verdict

judge_llm = OpenAICompatClient(
    best_cfg.llm.model_copy(
        update={
            "model": best_cfg.judge.model,
            "extra_body": best_cfg.judge.extra_body,
            "temperature": 0.0,
        }
    )
)
template = best_cfg.judge.prompt_template.read_text(encoding="utf-8")
rec = recs["aday"]
ruled = rule_verdict(rec)
print("kural kararı:", ruled.reason if ruled else "yok -> LLM'e gider")
verdict = judge_record(rec, judge_llm, template, best_cfg.judge.model, "")
print(f"karar: {verdict.correct} ({verdict.method}) - {verdict.reason}")

kural kararı: exact match with a reference answer
karar: True (rule) - exact match with a reference answer


Kaydedilmiş koşudan, F1'i düşük ama judge'ın doğru dediği örnekler: F1'in nerede haksız olduğunu gösterir.

In [16]:
RUN = "m1b_section_best"  # <- değiştir
verdicts = {
    v["question_id"]: v
    for v in map(json.loads, (RESULTS / RUN / "judge.jsonl").read_text().splitlines())
}
shown = 0
for r in runs[RUN].records:
    v = verdicts.get(r.question_id)
    if v and v["correct"] and v["method"] == "llm" and r.answer_f1 < 0.3 and shown < 3:
        shown += 1
        print(f"S: {r.question}\n  tahmin: {r.prediction[:160]}\n  gold  : {r.gold_texts[0][:160]}")
        print(f"  F1={r.answer_f1:.2f} | judge: {v['reason'][:160]}\n")

S: How does proposed qualitative annotation schema looks like?
  tahmin: The proposed qualitative annotation schema categorizes question, answer, and passage triples using a hierarchical framework (as shown in Figure 4) with abstract
  gold  : The resulting taxonomy of the framework is shown in Figure FIGREF10
  F1=0.12 | judge: The answer provides additional details but does not contradict the reference answers.

S: How does their ensemble method work?
  tahmin: We started with the best performing model according to validation performance. Then in each step we tried adding the best performing model that had not been pre
  gold  : simply averaging the predictions from the constituent single models
  F1=0.19 | judge: The answer correctly states that the ensemble method works by averaging the predictions from the constituent single models.

S: Which baselines did they compare against?
  tahmin: Latent Syntax Tree-LSTM (BIBREF10), Tree-based CNN (BIBREF35), Gumbel Tree-LSTM (BIBREF11), NS

## 10 · Soru bazlı karşılaştırma: baseline → aday

Ortalama bir sayı neyin değiştiğini söylemez. Aynı soruların iki koşudaki kayıtlarını (`records.jsonl`) karşılaştırınca kazancın nereden geldiği görünür: kanıt skoru artan soruların çoğunda cevap aynı kalmış.

In [17]:
best = {r.question_id: r for r in runs["m1b_section_best"].records}
pairs = [(b, best[b.question_id]) for b in base if b.question_id in best]
for field in ("answer_f1", "evidence_f1"):
    up = sum(getattr(n, field) > getattr(o, field) + 1e-9 for o, n in pairs)
    down = sum(getattr(n, field) < getattr(o, field) - 1e-9 for o, n in pairs)
    print(
        f"{field:12}: {up:3} soruda arttı, {down:3} soruda azaldı, {len(pairs) - up - down:3} soruda aynı"
    )

print("\nKanıt skoru en çok artan 5 soru:")
for o, n in sorted(pairs, key=lambda p: p[1].evidence_f1 - p[0].evidence_f1, reverse=True)[:5]:
    print(
        f"  ev {o.evidence_f1:.2f}->{n.evidence_f1:.2f} | ans {o.answer_f1:.2f}->{n.answer_f1:.2f} | {o.question[:70]}"
    )

answer_f1   :  70 soruda arttı,  58 soruda azaldı, 122 soruda aynı
evidence_f1 : 119 soruda arttı,  40 soruda azaldı,  91 soruda aynı

Kanıt skoru en çok artan 5 soru:
  ev 0.00->1.00 | ans 0.00->0.50 | what was the baseline?
  ev 0.00->1.00 | ans 0.25->0.00 | What alternative to Gibbs sampling is used?
  ev 0.00->1.00 | ans 0.21->0.95 | How are models evaluated in this human-machine communication game?
  ev 0.00->1.00 | ans 0.00->0.53 | What is the additive modification to the objective function?
  ev 0.00->1.00 | ans 0.22->0.83 | Which dataset has been used in this work?


## Bitti: sırada ne var

- **Karar:** M2'nin karşılaştırma noktası = bölüm chunking + paragraf atfı (`docs/decisions/002-m1b-pipeline.md`). Özet dosyası: `docs/m1b_ozet_tr.md`.
- **M2:** LangGraph ile agentic pipeline ve **korpus modu** (soru hangi makalede bilinmeden 281 makalenin hepsinde arama). Reranker ve multi-query orada yeniden denenecek, çünkü aday havuzu artık tek bir makale değil.
- Kendin dene: `QUESTION_INDEX`'i 21 veya 45 yap (kanıtı en çok artan sorular), ya da 1. adımda `RUNS`'a kendi koşunu ekle. Terminalden: `uv run paperlens trace --config experiments/configs/m1b_section_best.yaml --index 16`.

In [18]:
for b in bundles.values():
    b.dense.close()  # Qdrant yerel dosya kilidini bırak